# Colab Adaptive Gold Builder: V4 / V5 / V6

This notebook builds the next adaptive recommendation data layer from the existing Two-Tower Gold dataset.

Input Gold root:

```text
/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab/
```

Output adaptive root:

```text
/content/drive/MyDrive/recsys/data/gold/adaptive/v1/
```

It creates reusable point-in-time sequence examples for future model notebooks:

```text
sequence_examples/{train,validation,test}
  full point-in-time examples with recent user history arrays

model_views/v4_sequence_retrieval/contract.json
  retrieval contract: short-term sequence + long-term user + item tower features

model_views/v5_sequence_ranker/contract.json
  ranker contract: same examples plus richer labels/outcome targets

model_views/v6_online_state/contract.json
  online/adaptive serving simulation contract

event_stream/
  chronological interaction stream for state updates

serving_queries/{train,validation,test}
  lightweight request/query rows for online simulation and later serving notebooks
```

Silver data is not required. This notebook does not train a model.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import importlib.util
import subprocess
import sys

missing = []
for package in ['pyspark']:
    if importlib.util.find_spec(package) is None:
        missing.append(package)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.6'])
else:
    print('PySpark is already available.')

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

GOLD_ROOT = Path('/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab')
ADAPTIVE_ROOT = Path('/content/drive/MyDrive/recsys/data/gold/adaptive/v1')

# Conservative defaults for standard Colab RAM. Increase to 50/20 on High-RAM if needed.
MAX_SEQ_LEN = 30
MAX_SESSION_SEQ_LEN = 10
OVERWRITE = True
SHUFFLE_PARTITIONS = 96
DRIVER_MEMORY = '10g'

# Set this to a small number, e.g. 20, for a quick dry run.
# Keep None for the final full build.
DEVELOPMENT_USER_LIMIT = None

SPLITS = ['train', 'validation', 'test']

print('Gold root:', GOLD_ROOT, GOLD_ROOT.exists())
print('Adaptive output:', ADAPTIVE_ROOT)

In [ ]:
required_paths = {
    'targets/train': GOLD_ROOT / 'targets/train',
    'targets/validation': GOLD_ROOT / 'targets/validation',
    'targets/test': GOLD_ROOT / 'targets/test',
    'user_state/train': GOLD_ROOT / 'user_state/train',
    'item_features/point_in_time/train': GOLD_ROOT / 'item_features/point_in_time/train',
    'history/events': GOLD_ROOT / 'history/events',
    'transforms/numeric_stats.json': GOLD_ROOT / 'transforms/numeric_stats.json',
    'feature_catalog.json': GOLD_ROOT / 'feature_catalog.json',
    'manifest.json': GOLD_ROOT / 'manifest.json',
}

for name, path in required_paths.items():
    if path.is_dir():
        n = len(list(path.rglob('part-*.parquet')))
        status = f'{n} parquet files'
    else:
        status = 'OK' if path.exists() else 'MISSING'
    print(f'{name:38s} {status}')

## Spark Session

The build uses Spark because sequence construction is a window operation over the interaction stream. The important rule is point-in-time safety: each training row only receives events strictly before its `as_of_time`.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('adaptive-gold-builder-v1')
    .master('local[2]')
    .config('spark.driver.memory', DRIVER_MEMORY)
    .config('spark.sql.shuffle.partitions', str(SHUFFLE_PARTITIONS))
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version)
print('Master:', spark.sparkContext.master)

## Feature Contracts

These contracts are saved next to the data so later model notebooks can read the same feature definitions without guessing.

In [ ]:
USER_ID_COLUMNS = ['user_id']
ITEM_ID_COLUMNS = ['video_id']

USER_CATEGORICAL = ['user_active_degree']
ITEM_CATEGORICAL = ['video_type', 'upload_type']

USER_NUMERIC = [
    'is_lowactive_period', 'is_live_streamer', 'is_video_author',
    'follow_user_num', 'fans_user_num', 'friend_user_num', 'register_days',
    *[f'onehot_feat{i}' for i in range(18)],
    'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate',
    'user_hist_comment_rate', 'user_hist_forward_rate', 'user_hist_follow_rate',
    'user_hist_hate_rate', 'user_hist_avg_watch_ratio', 'user_hist_avg_play_time_sec',
    'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate',
    'session_prior_like_rate', 'session_prior_hate_rate', 'session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta',
    'event_hour', 'event_dayofweek', 'tab', 'is_rand',
]

ITEM_NUMERIC = [
    'video_duration_sec', 'aspect_ratio', 'visible_status', 'music_type',
    'upload_age_days_at_event', 'item_hist_events', 'item_hist_long_view_rate',
    'item_hist_like_rate', 'item_hist_hate_rate', 'item_hist_avg_watch_ratio',
]

SEQUENCE_COLUMNS = [
    'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
    'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks',
    'recent_time_ms', 'recent_event_count', 'seconds_since_last_event',
    'recent_long_view_rate', 'recent_like_rate', 'recent_hate_rate', 'recent_avg_engagement_strength',
    'session_recent_video_ids', 'session_recent_target_classes', 'session_recent_engagement_strengths',
    'session_recent_long_views', 'session_recent_likes', 'session_recent_hates', 'session_recent_clicks',
    'session_recent_time_ms', 'session_recent_event_count', 'session_seconds_since_last_event',
]

TARGET_COLUMNS = [
    'target_class', 'is_positive', 'is_observed_negative', 'is_ambiguous',
    'engagement_strength', 'watch_ratio_clipped', 'long_view', 'is_like',
    'is_comment', 'is_forward', 'is_follow', 'is_hate', 'is_click',
    'is_profile_enter', 'play_time_ms', 'duration_ms', 'watch_ratio',
]

print('user numeric:', len(USER_NUMERIC))
print('item numeric:', len(ITEM_NUMERIC))
print('sequence columns:', len(SEQUENCE_COLUMNS))

## Build Point-In-Time Recent History

For each observed event, this creates two histories:

- `recent_*`: last `MAX_SEQ_LEN` events from the same user before the current event.
- `session_recent_*`: last `MAX_SESSION_SEQ_LEN` events from the same user and same session before the current event.

Current-event outcome fields are not copied into the recent arrays until the next event, so the sequence side remains point-in-time safe.

In [ ]:
def read_parquet(path: Path):
    return spark.read.parquet(str(path))

history = read_parquet(GOLD_ROOT / 'history/events')

if DEVELOPMENT_USER_LIMIT is not None:
    keep_users = history.select('user_id').distinct().limit(DEVELOPMENT_USER_LIMIT)
    history = history.join(keep_users, on='user_id', how='inner')

history = history.select(
    'event_id', 'user_id', 'video_id', 'event_ts', 'time_ms', 'session_id',
    'user_event_index', 'session_event_index', 'target_class', 'engagement_strength',
    'long_view', 'is_like', 'is_comment', 'is_forward', 'is_follow', 'is_hate',
    'is_click', 'is_profile_enter',
)

user_window = Window.partitionBy('user_id').orderBy('time_ms', 'event_id')
session_window = Window.partitionBy('user_id', 'session_id').orderBy('time_ms', 'event_id')

event_struct = F.struct(
    F.col('video_id').alias('video_id'),
    F.col('target_class').alias('target_class'),
    F.col('engagement_strength').alias('engagement_strength'),
    F.col('long_view').alias('long_view'),
    F.col('is_like').alias('is_like'),
    F.col('is_hate').alias('is_hate'),
    F.col('is_click').alias('is_click'),
    F.col('time_ms').alias('time_ms'),
)

def lagged_event_array(window, max_len):
    # Oldest -> newest order, with null lag positions removed below.
    return F.array(*[F.lag(event_struct, offset).over(window) for offset in range(max_len, 0, -1)])

sequence_context = (
    history
    .withColumn('recent_events', lagged_event_array(user_window, MAX_SEQ_LEN))
    .withColumn('session_recent_events', lagged_event_array(session_window, MAX_SESSION_SEQ_LEN))
    .withColumn('recent_events', F.expr('filter(recent_events, x -> x is not null)'))
    .withColumn('session_recent_events', F.expr('filter(session_recent_events, x -> x is not null)'))
    .withColumn('recent_video_ids', F.expr('transform(recent_events, x -> x.video_id)'))
    .withColumn('recent_target_classes', F.expr('transform(recent_events, x -> x.target_class)'))
    .withColumn('recent_engagement_strengths', F.expr('transform(recent_events, x -> x.engagement_strength)'))
    .withColumn('recent_long_views', F.expr('transform(recent_events, x -> x.long_view)'))
    .withColumn('recent_likes', F.expr('transform(recent_events, x -> x.is_like)'))
    .withColumn('recent_hates', F.expr('transform(recent_events, x -> x.is_hate)'))
    .withColumn('recent_clicks', F.expr('transform(recent_events, x -> x.is_click)'))
    .withColumn('recent_time_ms', F.expr('transform(recent_events, x -> x.time_ms)'))
    .withColumn('session_recent_video_ids', F.expr('transform(session_recent_events, x -> x.video_id)'))
    .withColumn('session_recent_target_classes', F.expr('transform(session_recent_events, x -> x.target_class)'))
    .withColumn('session_recent_engagement_strengths', F.expr('transform(session_recent_events, x -> x.engagement_strength)'))
    .withColumn('session_recent_long_views', F.expr('transform(session_recent_events, x -> x.long_view)'))
    .withColumn('session_recent_likes', F.expr('transform(session_recent_events, x -> x.is_like)'))
    .withColumn('session_recent_hates', F.expr('transform(session_recent_events, x -> x.is_hate)'))
    .withColumn('session_recent_clicks', F.expr('transform(session_recent_events, x -> x.is_click)'))
    .withColumn('session_recent_time_ms', F.expr('transform(session_recent_events, x -> x.time_ms)'))
    .withColumn('recent_event_count', F.size('recent_video_ids'))
    .withColumn('session_recent_event_count', F.size('session_recent_video_ids'))
    .withColumn('recent_last_time_ms', F.expr('try_element_at(recent_time_ms, -1)'))
    .withColumn('session_recent_last_time_ms', F.expr('try_element_at(session_recent_time_ms, -1)'))
    .withColumn('seconds_since_last_event', (F.col('time_ms') - F.col('recent_last_time_ms')) / F.lit(1000.0))
    .withColumn('session_seconds_since_last_event', (F.col('time_ms') - F.col('session_recent_last_time_ms')) / F.lit(1000.0))
    .withColumn('recent_long_view_rate', F.expr('aggregate(recent_long_views, 0D, (acc, x) -> acc + coalesce(double(x), 0D)) / greatest(size(recent_long_views), 1)'))
    .withColumn('recent_like_rate', F.expr('aggregate(recent_likes, 0D, (acc, x) -> acc + coalesce(double(x), 0D)) / greatest(size(recent_likes), 1)'))
    .withColumn('recent_hate_rate', F.expr('aggregate(recent_hates, 0D, (acc, x) -> acc + coalesce(double(x), 0D)) / greatest(size(recent_hates), 1)'))
    .withColumn('recent_avg_engagement_strength', F.expr('aggregate(recent_engagement_strengths, 0D, (acc, x) -> acc + coalesce(double(x), 0D)) / greatest(size(recent_engagement_strengths), 1)'))
    .select(
        F.col('user_id').alias('seq_user_id'),
        F.col('video_id').alias('current_video_id'),
        F.col('event_ts').alias('current_event_ts'),
        F.col('time_ms').alias('current_time_ms'),
        F.col('event_id').alias('current_event_id'),
        'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
        'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks',
        'recent_time_ms', 'recent_event_count', 'seconds_since_last_event',
        'recent_long_view_rate', 'recent_like_rate', 'recent_hate_rate', 'recent_avg_engagement_strength',
        'session_recent_video_ids', 'session_recent_target_classes', 'session_recent_engagement_strengths',
        'session_recent_long_views', 'session_recent_likes', 'session_recent_hates', 'session_recent_clicks',
        'session_recent_time_ms', 'session_recent_event_count', 'session_seconds_since_last_event',
    )
    .dropDuplicates(['seq_user_id', 'current_video_id', 'current_event_ts'])
)

print('sequence context schema:')
sequence_context.printSchema()

# Materialize once to break the expensive window lineage before building splits.
sequence_context_path = ADAPTIVE_ROOT / '_intermediate' / 'sequence_context'
sequence_context.write.mode('overwrite' if OVERWRITE else 'errorifexists').parquet(str(sequence_context_path))
sequence_context = read_parquet(sequence_context_path)
print('materialized sequence_context:', sequence_context_path)

## Build Adaptive Examples By Split

The physical `sequence_examples/{split}` table is the source of truth for V4/V5 model notebooks.

It joins:

```text
targets/{split}
+ user_state/{split}
+ item_features/point_in_time/{split}
+ sequence_context from history/events
```

Join to history uses `(user_id, video_id, as_of_time == event_ts)`, so sequence history is attached to the current interaction without an expensive all-prior-events join.

In [ ]:
def select_available(df, columns):
    existing = set(df.columns)
    return [c for c in columns if c in existing]


def build_split_examples(split: str):
    targets = read_parquet(GOLD_ROOT / 'targets' / split)
    user_state = read_parquet(GOLD_ROOT / 'user_state' / split)
    item_features = read_parquet(GOLD_ROOT / 'item_features' / 'point_in_time' / split)

    if DEVELOPMENT_USER_LIMIT is not None:
        keep_users = history.select('user_id').distinct().limit(DEVELOPMENT_USER_LIMIT)
        targets = targets.join(keep_users, on='user_id', how='inner')
        user_state = user_state.join(keep_users, on='user_id', how='inner')

    user_keep = ['example_id', 'user_id', 'session_id', *USER_CATEGORICAL, *USER_NUMERIC]
    item_keep = ['example_id', 'video_id', *ITEM_CATEGORICAL, *ITEM_NUMERIC]
    user_state = user_state.select(*select_available(user_state, user_keep))
    item_features = item_features.select(*select_available(item_features, item_keep))

    base = (
        targets
        .join(user_state, on=['example_id', 'user_id'], how='left')
        .join(item_features, on=['example_id', 'video_id'], how='left')
        .withColumn('split', F.lit(split))
        .withColumn('retrieval_label', F.col('is_positive').cast('int'))
        .withColumn('ranker_label_strength', F.col('engagement_strength').cast('double'))
    )

    enriched = (
        base.alias('b')
        .join(
            sequence_context.alias('s'),
            (F.col('b.user_id') == F.col('s.seq_user_id'))
            & (F.col('b.video_id') == F.col('s.current_video_id'))
            & (F.col('b.as_of_time') == F.col('s.current_event_ts')),
            how='left',
        )
        .drop('seq_user_id')
        .drop('current_video_id')
        .drop('current_event_ts')
    )

    # Make first-event histories explicit empty arrays instead of null where practical.
    array_defaults = {
        'recent_video_ids': 'array<int>',
        'recent_target_classes': 'array<string>',
        'recent_engagement_strengths': 'array<double>',
        'recent_long_views': 'array<int>',
        'recent_likes': 'array<int>',
        'recent_hates': 'array<int>',
        'recent_clicks': 'array<int>',
        'recent_time_ms': 'array<long>',
        'session_recent_video_ids': 'array<int>',
        'session_recent_target_classes': 'array<string>',
        'session_recent_engagement_strengths': 'array<double>',
        'session_recent_long_views': 'array<int>',
        'session_recent_likes': 'array<int>',
        'session_recent_hates': 'array<int>',
        'session_recent_clicks': 'array<int>',
        'session_recent_time_ms': 'array<long>',
    }
    for col_name, col_type in array_defaults.items():
        if col_name in enriched.columns:
            enriched = enriched.withColumn(col_name, F.coalesce(F.col(col_name), F.expr(f'cast(array() as {col_type})')))

    for col_name in ['recent_event_count', 'session_recent_event_count']:
        if col_name in enriched.columns:
            enriched = enriched.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0)))

    return enriched


def write_parquet(df, path: Path):
    mode = 'overwrite' if OVERWRITE else 'errorifexists'
    (
        df.repartition(SHUFFLE_PARTITIONS, 'user_id')
        .write.mode(mode)
        .parquet(str(path))
    )

In [ ]:
ADAPTIVE_ROOT.mkdir(parents=True, exist_ok=True)

split_summaries = []
for split in SPLITS:
    print(f'\n=== Building {split} ===')
    examples = build_split_examples(split)

    output_path = ADAPTIVE_ROOT / 'sequence_examples' / split
    write_parquet(examples, output_path)
    print('wrote:', output_path)

    summary = (
        examples
        .agg(
            F.count('*').alias('examples'),
            F.countDistinct('user_id').alias('users'),
            F.countDistinct('video_id').alias('items'),
            F.avg('recent_event_count').alias('avg_recent_events'),
            F.avg('session_recent_event_count').alias('avg_session_recent_events'),
            F.sum('is_positive').alias('positive_examples'),
            F.sum('is_observed_negative').alias('observed_negative_examples'),
            F.sum('is_ambiguous').alias('ambiguous_examples'),
        )
        .withColumn('split', F.lit(split))
        .collect()[0]
        .asDict()
    )
    split_summaries.append(summary)
    print(summary)

## Build V6 Online State Inputs

`event_stream/` is the ordered stream for simulating near-real-time state updates. Later V6 notebooks can replay it and update a user's short-term state after each interaction.

`serving_queries/{split}` is a lightweight table of candidate request rows. It points back to `sequence_examples` through `example_id`.

In [ ]:
event_stream = (
    history
    .withColumn('event_hour', F.hour('event_ts'))
    .withColumn('event_dayofweek', F.dayofweek('event_ts'))
    .select(
        'event_id', 'user_id', 'video_id', 'event_ts', 'time_ms', 'session_id',
        'user_event_index', 'session_event_index', 'target_class', 'engagement_strength',
        'long_view', 'is_like', 'is_comment', 'is_forward', 'is_follow', 'is_hate',
        'is_click', 'is_profile_enter', 'event_hour', 'event_dayofweek',
    )
)

write_parquet(event_stream, ADAPTIVE_ROOT / 'event_stream')
print('wrote:', ADAPTIVE_ROOT / 'event_stream')

serving_summaries = []
for split in SPLITS:
    examples = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / split)
    serving_queries = examples.select(
        'split', 'example_id', 'context_id', 'user_id', 'video_id', 'session_id', 'as_of_time',
        'target_class', 'is_positive', 'is_observed_negative', 'is_ambiguous',
        'engagement_strength', 'retrieval_label', 'ranker_label_strength',
        'recent_event_count', 'session_recent_event_count',
    )
    path = ADAPTIVE_ROOT / 'serving_queries' / split
    write_parquet(serving_queries, path)
    summary = serving_queries.agg(F.count('*').alias('queries')).withColumn('split', F.lit(split)).collect()[0].asDict()
    serving_summaries.append(summary)
    print('wrote:', path, summary)

## Leakage Checks

These checks should stay clean:

- the last event in `recent_time_ms` must be strictly earlier than the current event time;
- sequence length must not exceed the configured max;
- unmatched sequence rows should mostly be first events or timestamp/key mismatches worth inspecting.

In [ ]:
check_rows = []
for split in SPLITS:
    df = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / split)
    check = (
        df
        .withColumn('last_recent_time_ms', F.expr('try_element_at(recent_time_ms, -1)'))
        .agg(
            F.count('*').alias('examples'),
            F.sum(F.when(F.col('last_recent_time_ms') >= F.col('current_time_ms'), 1).otherwise(0)).alias('leakage_violations'),
            F.max('recent_event_count').alias('max_recent_event_count'),
            F.max('session_recent_event_count').alias('max_session_recent_event_count'),
            F.avg(F.when(F.col('recent_event_count') > 0, 1).otherwise(0)).alias('has_recent_history_rate'),
        )
        .withColumn('split', F.lit(split))
        .collect()[0]
        .asDict()
    )
    check_rows.append(check)

for row in check_rows:
    print(row)

## Save Contracts And Manifest

The model contracts make the data layer explicit:

- V4: sequence-aware retrieval candidate generation.
- V5: sequence-aware ranker with richer labels/outcomes.
- V6: near-real-time state update simulation and serving queries.

In [ ]:
contracts_root = ADAPTIVE_ROOT / 'model_views'
contracts_root.mkdir(parents=True, exist_ok=True)

common_source = 'sequence_examples/{split}'

contracts = {
    'v4_sequence_retrieval': {
        'task': 'adaptive retrieval / candidate generation',
        'source_table': common_source,
        'label_columns': ['retrieval_label', 'target_class', 'is_positive', 'is_observed_negative', 'is_ambiguous'],
        'recommended_train_filter': "target_class in ('STRONG_POSITIVE', 'OBSERVED_NEGATIVE') or use sampled negatives",
        'user_inputs': USER_ID_COLUMNS + USER_CATEGORICAL + USER_NUMERIC + SEQUENCE_COLUMNS,
        'item_inputs': ITEM_ID_COLUMNS + ITEM_CATEGORICAL + ITEM_NUMERIC,
        'notes': [
            'Use recent_* arrays for short-term preference encoding.',
            'Keep long-term user_state features for stability.',
            'Use point-in-time candidate item features.',
        ],
    },
    'v5_sequence_ranker': {
        'task': 'adaptive ranker / engagement prediction',
        'source_table': common_source,
        'label_columns': TARGET_COLUMNS + ['ranker_label_strength', 'retrieval_label'],
        'recommended_objectives': [
            'binary long_view or STRONG_POSITIVE classification',
            'engagement_strength regression/ranking',
            'multi-task heads for like/follow/hate if needed',
        ],
        'user_inputs': USER_ID_COLUMNS + USER_CATEGORICAL + USER_NUMERIC + SEQUENCE_COLUMNS,
        'item_inputs': ITEM_ID_COLUMNS + ITEM_CATEGORICAL + ITEM_NUMERIC,
        'notes': [
            'V5 can use all target classes, not just positives.',
            'Avoid using current-event outcome columns as inputs; they are labels only.',
        ],
    },
    'v6_online_state': {
        'task': 'near-real-time user state simulation and serving',
        'event_stream_table': 'event_stream',
        'serving_query_table': 'serving_queries/{split}',
        'state_update_inputs': [
            'event_id', 'user_id', 'video_id', 'event_ts', 'time_ms', 'session_id',
            'target_class', 'engagement_strength', 'long_view', 'is_like', 'is_hate', 'is_click',
        ],
        'serving_inputs': USER_ID_COLUMNS + ['as_of_time', 'session_id', 'recent_event_count', 'session_recent_event_count'],
        'notes': [
            'Replay event_stream ordered by time_ms to update short-term user state.',
            'Use serving_queries to evaluate whether the state before each request ranks the observed item well.',
            'This is the bridge from offline sequence data to near-real-time adaptive serving.',
        ],
    },
}

for name, payload in contracts.items():
    out_dir = contracts_root / name
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / 'contract.json', 'w') as f:
        json.dump(payload, f, indent=2)
    print('wrote:', out_dir / 'contract.json')

manifest = {
    'adaptive_dataset_version': 'adaptive_v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_gold_root': str(GOLD_ROOT),
    'output_root': str(ADAPTIVE_ROOT),
    'max_seq_len': MAX_SEQ_LEN,
    'max_session_seq_len': MAX_SESSION_SEQ_LEN,
    'development_user_limit': DEVELOPMENT_USER_LIMIT,
    'splits': SPLITS,
    'split_summaries': split_summaries,
    'serving_summaries': serving_summaries,
    'leakage_checks': check_rows,
    'artifact_paths': {
        'sequence_examples': str(ADAPTIVE_ROOT / 'sequence_examples'),
        'event_stream': str(ADAPTIVE_ROOT / 'event_stream'),
        'serving_queries': str(ADAPTIVE_ROOT / 'serving_queries'),
        'model_views': str(ADAPTIVE_ROOT / 'model_views'),
        'manifest': str(ADAPTIVE_ROOT / 'manifest.json'),
    },
}

with open(ADAPTIVE_ROOT / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2)[:5000])

## Quick Preview For Model Notebooks

This preview shows what a future V4/V5/V6 notebook will load.

In [ ]:
preview = read_parquet(ADAPTIVE_ROOT / 'sequence_examples' / 'train')
preview.select(
    'example_id', 'user_id', 'video_id', 'as_of_time', 'target_class', 'retrieval_label',
    'recent_event_count', 'recent_video_ids', 'recent_target_classes',
    'session_recent_event_count', 'session_recent_video_ids',
).show(5, truncate=80)

print('Adaptive data written to:', ADAPTIVE_ROOT)